# 05 — End to End

The whole flow in one notebook: anvil + deploy + provider FastAPI + consumer FastAPI + real Ollama.

Prereqs: `anvil`, `forge`, and `ollama serve` running with `llama3.2:3b` pulled.

## Setup

In [ ]:
import sys, pathlib
_ROOT = pathlib.Path.cwd().resolve()
if (_ROOT / 'shared').is_dir():
    sys.path.insert(0, str(_ROOT))
elif (_ROOT.parent / 'shared').is_dir():
    sys.path.insert(0, str(_ROOT.parent))


In [ ]:
import os, socket, threading, time, asyncio
import httpx
import uvicorn
from shared.anvil import anvil
from shared.config import Config
from shared.deploy import deploy_contracts

DEPLOYER = '0xac0974bec39a17e36ba4a6b4d238ff944bacb478cbed5efcae784d7bf4f2ff80'
PROVIDER = '0x59c6995e998f97a5a0044966f0945389dc9e86dae88c7a8412f4603b6b78690d'
CONSUMER = '0x5de4111afa1a4b94908f83103eb1f1706367c2e68ca870fc3fb9a804cdab365a'

def free_port():
    with socket.socket() as s:
        s.bind(('127.0.0.1', 0))
        return s.getsockname()[1]

def serve(app, port):
    cfg = uvicorn.Config(app, host='127.0.0.1', port=port,
                          log_level='warning', lifespan='on')
    server = uvicorn.Server(cfg)
    t = threading.Thread(target=server.run, daemon=True)
    t.start()
    deadline = time.monotonic() + 10
    while time.monotonic() < deadline and not server.started:
        time.sleep(0.05)
    return server, t

## Build — anvil + deploy

In [ ]:
ctx = anvil(port=18545)
rpc_url = ctx.__enter__()
cfg = Config(rpc_url=rpc_url, deployer_private_key=DEPLOYER,
             provider_private_key=PROVIDER,
             consumer_private_key=CONSUMER, sdn_mock=True)
addrs = deploy_contracts(cfg)
print('deployed:', addrs)

## Build — provider + consumer FastAPI in-process

In [ ]:
provider_port = free_port()
consumer_port = free_port()
provider_url = f'http://127.0.0.1:{provider_port}'
consumer_url = f'http://127.0.0.1:{consumer_port}'

os.environ.update({
    'RPC_URL': rpc_url,
    'CONSUMER_PRIVATE_KEY': CONSUMER,
    'PROVIDER_PRIVATE_KEY': PROVIDER,
    'PROVIDER_BASE_URL': provider_url,
    'CONSUMER_BASE_URL': consumer_url,
    'PROVIDER_A2A_URLS': provider_url,
    'SDN_MOCK': 'true',
    'OLLAMA_HOST': os.environ.get('OLLAMA_HOST', 'http://127.0.0.1:11434'),
    'OLLAMA_MODEL': os.environ.get('OLLAMA_MODEL', 'llama3.2:3b'),
})

from provider.app import app as provider_app
from consumer.app import app as consumer_app
ps, pt = serve(provider_app, provider_port)
cs, ct = serve(consumer_app, consumer_port)
print('provider:', provider_url)
print('consumer:', consumer_url)

## Run — one negotiation

In [ ]:
async with httpx.AsyncClient(timeout=120.0) as http:
    resp = await http.post(f'{consumer_url}/chat',
                           json={'message': 'I need 5 Mbps for 10 minutes'})
    body = resp.json()
print('response:', body['response'])
for entry in body['log']:
    print(' ', entry['from'], '|', entry['message'])

## Inspect — on-chain events

In [ ]:
async with httpx.AsyncClient(timeout=10.0) as http:
    events = (await http.get(f'{consumer_url}/chain_events')).json()
for e in events:
    print(e['event'], '@ block', e['block'])

## Teardown

In [ ]:
ps.should_exit = True; cs.should_exit = True
pt.join(timeout=5); ct.join(timeout=5)
ctx.__exit__(None, None, None)
print('done')